<a href="https://colab.research.google.com/github/sachabidermann/airlift-planner/blob/main/BestModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
!unzip -o -j ames.zip

Archive:  ames.zip
  inflating: train.csv               
  inflating: sample_soln.csv         
  inflating: ._sample_soln.csv       
  inflating: solns.csv               
  inflating: ._solns.csv             
  inflating: test.csv                


In [ ]:
df = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
X = df.drop(columns = ['Id', 'SalePrice'])
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 22)

baseline = np.ones(len(y_test))*y_train.mean()
np.sqrt(mean_squared_error(y_test, baseline))

np.float64(78119.87384283314)

In [ ]:
num_cols = X.select_dtypes(['float', 'int']).columns.tolist()
cat_cols = X.select_dtypes('object').columns.tolist()

small_num = ['OverallQual', 'GrLivArea', 'TotalBsmtSF', 'GarageCars', 'YearBuilt', 'YearRemodAdd', 'FullBath']
small_cat = ['Neighborhood', 'KitchenQual', 'ExterQual']

big_num = ['OverallQual', 'GrLivArea', 'TotalBsmtSF', 'GarageCars', 'GarageArea', 'YearBuilt', 'YearRemodAdd', 'FullBath', 'OverallCond', 'LotArea', 'Fireplaces', 'BsmtFinSF1', '1stFlrSF', '2ndFlrSF', 'TotRmsAbvGrd', 'MasVnrArea']
big_cat = ['Neighborhood', 'KitchenQual', 'ExterQual', 'BsmtQual', 'SaleCondition', 'MSZoning', 'CentralAir', 'BldgType']

In [ ]:
cat_pipe = Pipeline([('impute', SimpleImputer(strategy = 'most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))])

In [ ]:
transformer_a = make_column_transformer((SimpleImputer(), small_num),
                                        (cat_pipe, small_cat))

pipe_a = Pipeline([('transform', transformer_a), ('model', LinearRegression())])
pipe_a.fit(X_train, y_train)

print(np.sqrt(mean_squared_error(y_train, pipe_a.predict(X_train))))
print(np.sqrt(mean_squared_error(y_test, pipe_a.predict(X_test))))

32508.99964407065
28583.20933804332


In [ ]:
transformer_b = make_column_transformer((SimpleImputer(), big_num),
                                        (cat_pipe, big_cat))

pipe_b = Pipeline([('transform', transformer_b), ('model', LinearRegression())])
pipe_b.fit(X_train, y_train)

print(np.sqrt(mean_squared_error(y_train, pipe_b.predict(X_train))))
print(np.sqrt(mean_squared_error(y_test, pipe_b.predict(X_test))))

27569.359661506627
26351.34992138496


In [ ]:
num_pipe = Pipeline([('impute', SimpleImputer()), ('scale', StandardScaler())])

transformer_c = make_column_transformer((num_pipe, small_num),
                                        (cat_pipe, small_cat))

for k in range(1, 21):
    pipe_c = Pipeline([('transform', transformer_c), ('model', KNeighborsRegressor(n_neighbors = k))])
    pipe_c.fit(X_train, y_train)
    print(k, np.sqrt(mean_squared_error(y_test, pipe_c.predict(X_test))))

1 38800.33702108785
2 33061.9653168108
3 30320.236795035704
4 29244.74564981004
5 28989.012267376984
6 28612.541552845905
7 28439.12044633034
8 28906.391584976674
9 29168.269125075614
10 29180.599835034805
11 29290.20644505779
12 30038.21661464035
13 29811.914484146753
14 30049.714930372742
15 29835.70081976041
16 29935.5908558368
17 30258.573797732915
18 30500.440103358505
19 30835.645857235482
20 30725.237416915377


In [ ]:
mask = df['GrLivArea'] < 4000

best = pipe_b
best.fit(X[mask], y[mask])

X_final = test[X.columns.tolist()]
preds = best.predict(X_final)

nn = Pipeline([('transform', transformer_c), ('model', KNeighborsRegressor(n_neighbors = 1))])
nn.fit(X, y)

outside = X_final['GrLivArea'] > X['GrLivArea'].max()
preds[outside] = nn.predict(X_final[outside])

pdf = pd.DataFrame(preds, columns = ['SalePrice'])
pdf['Id'] = test.index
pdf.to_csv('preds.csv', index = False)
pdf.head()

,SalePrice,Id
0,79086.778591,0
1,193263.865301,1
2,137697.940375,2
3,208587.590157,3
4,136980.998740,4


In [ ]:
print(pdf.shape)
pdf['SalePrice'].describe()

(260, 2)


,SalePrice
count,260.000000
mean,180008.487708
std,70695.916971
min,48448.372069
25%,131670.703182
50%,165486.073176
75%,219301.139688
max,453454.138679
